# The role of Temperature Scaling

In [2]:
import torch
import random
import pandas as pd

import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from calibration_methods.models.resnet import * 
from calibration_methods.models.mlp_mixer import * 
from calibration_methods.models.vit import * 
from calibration_methods.models.clip import * 

from calibration_methods.methods import * 
from calibration_methods.metrics import *

**Loading models checkpoints**

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

## ResNet
resnet_cifar10 = load_model(ResNet(num_classes=10), "calibration_methods/checkpoints/resnet152_cifar10.pt")
resnet_cifar100 = load_model(ResNet(num_classes=100), "calibration_methods/checkpoints/resnet152_cifar100.pt")

## MLPMixer
mlp_mixer_cifar10 = load_model(MLPMixer(num_classes=10), "calibration_methods/checkpoints/MLPMixer_b16_224_cifar10.pt")
mlp_mixer_cifar100 = load_model(MLPMixer(num_classes=100), "calibration_methods/checkpoints/MLPMixer_b16_224_cifar100.pt")

## ViT
vit_cifar10 = load_model(ViT(num_classes=10), "calibration_methods/checkpoints/ViT_cifar10.pt")
vit_cifar100 = load_model(ViT(num_classes=100), "calibration_methods/checkpoints/ViT_cifar100.pt")

## CLIP
CIFAR10_LABELS = datasets.CIFAR10(root="calibration_methods/datasets", train=False, download=False).classes
CIFAR100_LABELS = datasets.CIFAR100(root="calibration_methods/datasets", train=False, download=False).classes

clip_cifar10 = CLIPZeroShot(labels=CIFAR10_LABELS, device=device).to(device).eval()
clip_cifar100 = CLIPZeroShot(labels=CIFAR100_LABELS, device=device).to(device).eval()

**Loading datasets**

In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    
def get_cifar_loaders(
    dataset_name,
    transform,
    root="calibration_methods/datasets",
    batch_size=128,
    num_workers=4,
    seed=42  # We keep the same seed as for training phase
):

    seed_everything(seed)

    g = torch.Generator()
    g.manual_seed(seed)

    if dataset_name == "cifar10":
        Dataset = datasets.CIFAR10

    elif dataset_name == "cifar100":
        Dataset = datasets.CIFAR100

    train_dataset = Dataset(root=root, train=True, download=False, transform=transform)
    test_dataset = Dataset(root=root, train=False, download=False, transform=transform)

    train_set, val_set = random_split(train_dataset, [45000, 5000], generator=g)

    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=g
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=g
    )

    return  val_loader, test_loader


Defining transforms 

In [5]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073),
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

# ResNet
transform_cifar10 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_cifar100 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# MLPMixer
transform_mixer_cifar10 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_mixer_cifar100 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# ViT
transform_vit_cifar = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# CLIP
transform_clip_cifar = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

In [6]:
# ResNet 
val_loader_cifar10, test_loader_cifar10= get_cifar_loaders(
    dataset_name="cifar10",
    transform=transform_cifar10,
    batch_size=256
)

val_loader_cifar100, test_loader_cifar100= get_cifar_loaders(
    dataset_name="cifar100",
    transform=transform_cifar100,
    batch_size=256
)

# MLP Mixer
val_loader_mixer_cifar10, test_loader_mixer_cifar10= get_cifar_loaders(
    dataset_name="cifar10",
    transform=transform_mixer_cifar10,
    batch_size=256
)

val_loader_mixer_cifar100, test_loader_mixer_cifar100= get_cifar_loaders(
    dataset_name="cifar100",
    transform=transform_mixer_cifar100,
    batch_size=256
)

# ViT
val_loader_vit_cifar10, test_loader_vit_cifar10 = get_cifar_loaders(
    dataset_name="cifar10",
    transform=transform_vit_cifar,
    batch_size=256
)

val_loader_vit_cifar100, test_loader_vit_cifar100 = get_cifar_loaders(
    dataset_name="cifar100",
    transform=transform_vit_cifar,
    batch_size=256
)

# CLIP
val_loader_clip_cifar10, test_loader_clip_cifar10 = get_cifar_loaders(
    dataset_name="cifar10",
    transform=clip_cifar10.preprocess,
    batch_size=256
)

val_loader_clip_cifar100, test_loader_clip_cifar100 = get_cifar_loaders(
    dataset_name="cifar100",
    transform=clip_cifar100.preprocess,
    batch_size=256
)

## Calibration Methods

**Metrics for test data**

In [7]:
M = 10
bins = torch.linspace(0, 1, M + 1)

M_test = 20
bins_test = torch.linspace(0, 1, M_test + 1)

def get_labels(data_loader):
    all_labels = []

    for _, y in data_loader:
        all_labels.append(y)

    return torch.cat(all_labels)

# CIFAR10
true_labels_cifar10 = F.one_hot(get_labels(test_loader_cifar10), num_classes=10).float()

calibMetrics_cifar10 = CalibrationMetrics(
    bins=bins_test,
    true_labels=true_labels_cifar10,
    device=device
)

# CIFAR100
true_labels_cifar100 = F.one_hot(get_labels(test_loader_cifar100), num_classes=100).float()

calibMetrics_cifar100 = CalibrationMetrics(
    bins=bins_test,
    true_labels=true_labels_cifar100,
    device=device
)

In [8]:
def get_logits(model, dataloader, device):
    model.eval()
    all_logits = []

    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(device)
            logits = model(images)
            all_logits.append(logits.cpu())

    return torch.cat(all_logits, dim=0)

In [9]:
metrics_summary_cifar10 = dict()
metrics_summary_cifar100 = dict()

Functions for accuracy

In [10]:
def get_accuracy_from_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = logits.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

def get_accuracy_from_probs(probs, true_labels):
    preds = probs.argmax(dim=1)
    targets = true_labels.argmax(dim=1)

    correct = (preds == targets).sum().item()
    total = targets.size(0)

    return correct / total

accuracy_summary_cifar10 = dict()
accuracy_summary_cifar100 = dict()

### Uncalibrated models

In [11]:
def get_metrics_from_model(model, test_loader, dataset_name):
    logits = get_logits(model, test_loader, device)
    probs = torch.softmax(logits, dim=1)

    if dataset_name == 'cifar10':
        metrics = calibMetrics_cifar10.get_metrics(probs)
    else:
        metrics = calibMetrics_cifar100.get_metrics(probs)

    return metrics

In [12]:
metrics_summary_cifar10["Uncalibrated"] = {
    "ResNet" : get_metrics_from_model(resnet_cifar10, test_loader_cifar10, "cifar10"),
    "MLPMixer" : get_metrics_from_model(mlp_mixer_cifar10, test_loader_mixer_cifar10, "cifar10"),
    "ViT" : get_metrics_from_model(vit_cifar10, test_loader_vit_cifar10, "cifar10"),
    "CLIP" : get_metrics_from_model(clip_cifar10, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Uncalibrated"] = {
    "ResNet" : get_metrics_from_model(resnet_cifar100, test_loader_cifar100, "cifar100"),
    "MLPMixer" : get_metrics_from_model(mlp_mixer_cifar100, test_loader_mixer_cifar100, "cifar100"),
    "ViT" : get_metrics_from_model(vit_cifar100, test_loader_vit_cifar100, "cifar100"),
    "CLIP" : get_metrics_from_model(clip_cifar100, test_loader_clip_cifar100, "cifar100"),
}

In [13]:
accuracy_summary_cifar10["Uncalibrated"] = {
    "ResNet" : get_accuracy_from_model(resnet_cifar10, test_loader_cifar10),
    "MLPMixer" : get_accuracy_from_model(mlp_mixer_cifar10, test_loader_mixer_cifar10),
    "ViT" : get_accuracy_from_model(vit_cifar10, test_loader_vit_cifar10),
    "CLIP" : get_accuracy_from_model(clip_cifar10, test_loader_clip_cifar10),
}

accuracy_summary_cifar100["Uncalibrated"] = {
    "ResNet" : get_accuracy_from_model(resnet_cifar100, test_loader_cifar100),
    "MLPMixer" : get_accuracy_from_model(mlp_mixer_cifar100, test_loader_mixer_cifar100),
    "ViT" : get_accuracy_from_model(vit_cifar100, test_loader_vit_cifar100),
    "CLIP" : get_accuracy_from_model(clip_cifar100, test_loader_clip_cifar100),
}

### Histogramm Binning

In [14]:
# Getting validation probs
def get_probs_validation(model, val_loader):
    logits = get_logits(model, val_loader, device)
    probs = torch.softmax(logits, dim=1)
    return probs

# ResNet
probs_val_resnet_cifar10 = get_probs_validation(resnet_cifar10, val_loader_cifar10)
probs_val_resnet_cifar100 = get_probs_validation(resnet_cifar100, val_loader_cifar100)

# MLPMixer
probs_val_mixer_cifar10 = get_probs_validation(mlp_mixer_cifar10, val_loader_mixer_cifar10)
probs_val_mixer_cifar100 = get_probs_validation(mlp_mixer_cifar100, val_loader_mixer_cifar100)

# ViT
probs_val_vit_cifar10 = get_probs_validation(vit_cifar10, val_loader_vit_cifar10)
probs_val_vit_cifar100 = get_probs_validation(vit_cifar100, val_loader_vit_cifar100)

# CLIP
probs_val_clip_cifar10 = get_probs_validation(clip_cifar10, val_loader_clip_cifar10)
probs_val_clip_cifar100 = get_probs_validation(clip_cifar100, val_loader_clip_cifar100)

In [15]:
# Fitting Histogram Binning
labels_val_cifar10 = get_labels(val_loader_cifar10)
labels_val_cifar100 = get_labels(val_loader_cifar100)

# ResNet
resnet_cifar10_hb = HistogrammBinningCalibrator(bins)
resnet_cifar10_hb.fit(probs_val_resnet_cifar10, labels_val_cifar10)

resnet_cifar100_hb = HistogrammBinningCalibrator(bins)
resnet_cifar100_hb.fit(probs_val_resnet_cifar100, labels_val_cifar100)

# MLPMixer
mixer_cifar10_hb = HistogrammBinningCalibrator(bins)
mixer_cifar10_hb.fit(probs_val_mixer_cifar10, labels_val_cifar10)

mixer_cifar100_hb = HistogrammBinningCalibrator(bins)
mixer_cifar100_hb.fit(probs_val_mixer_cifar100, labels_val_cifar100)

# ViT
vit_cifar10_hb = HistogrammBinningCalibrator(bins)
vit_cifar10_hb.fit(probs_val_vit_cifar10, labels_val_cifar10)

vit_cifar100_hb = HistogrammBinningCalibrator(bins)
vit_cifar100_hb.fit(probs_val_vit_cifar100, labels_val_cifar100)

# CLIP
clip_cifar10_hb = HistogrammBinningCalibrator(bins)
clip_cifar10_hb.fit(probs_val_clip_cifar10, labels_val_cifar10)

clip_cifar100_hb = HistogrammBinningCalibrator(bins)
clip_cifar100_hb.fit(probs_val_clip_cifar100, labels_val_cifar100)

In [16]:
def get_metrics_hb(model, calibrator, test_loader, dataset_name):
    logits = get_logits(model, test_loader, device)
    probs = torch.softmax(logits, dim=1)
    calibrated_probs = calibrator.calibrate(probs)

    if dataset_name == 'cifar10':
        metrics = calibMetrics_cifar10.get_metrics(calibrated_probs)
    else:
        metrics = calibMetrics_cifar100.get_metrics(calibrated_probs)

    return metrics


def get_accuracy_hb(model, calibrator, test_loader, true_labels):
    logits = get_logits(model, test_loader, device)
    probs = torch.softmax(logits, dim=1)
    calibrated_probs = calibrator.calibrate(probs)

    return get_accuracy_from_probs(calibrated_probs, true_labels)

In [17]:
metrics_summary_cifar10["Histogram Binning"] = {
    "ResNet" : get_metrics_hb(resnet_cifar10, resnet_cifar10_hb, test_loader_cifar10, "cifar10"),
    "MLPMixer" : get_metrics_hb(mlp_mixer_cifar10, mixer_cifar10_hb, test_loader_mixer_cifar10, "cifar10"),
    "ViT" : get_metrics_hb(vit_cifar10, vit_cifar10_hb, test_loader_vit_cifar10, "cifar10"),
    "CLIP" : get_metrics_hb(clip_cifar10, clip_cifar10_hb, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Histogram Binning"] = {
    "ResNet" : get_metrics_hb(resnet_cifar100, resnet_cifar100_hb, test_loader_cifar100, "cifar100"),
    "MLPMixer" : get_metrics_hb(mlp_mixer_cifar100, mixer_cifar100_hb, test_loader_mixer_cifar100, "cifar100"),
    "ViT" : get_metrics_hb(vit_cifar100, vit_cifar100_hb, test_loader_vit_cifar100, "cifar100"),
    "CLIP" : get_metrics_hb(clip_cifar100, clip_cifar100_hb, test_loader_clip_cifar100, "cifar100"),
}

In [18]:
accuracy_summary_cifar10["Histogram Binning"] = {
    "ResNet" : get_accuracy_hb(resnet_cifar10, resnet_cifar10_hb, test_loader_cifar10, true_labels_cifar10),
    "MLPMixer" : get_accuracy_hb(mlp_mixer_cifar10, mixer_cifar10_hb, test_loader_mixer_cifar10, true_labels_cifar10),
    "ViT" : get_accuracy_hb(vit_cifar10, vit_cifar10_hb, test_loader_vit_cifar10, true_labels_cifar10),
    "CLIP" : get_accuracy_hb(clip_cifar10, clip_cifar10_hb, test_loader_clip_cifar10, true_labels_cifar10),
}

accuracy_summary_cifar100["Histogram Binning"] = {
    "ResNet" : get_accuracy_hb(resnet_cifar100, resnet_cifar100_hb, test_loader_cifar100, true_labels_cifar100),
    "MLPMixer" : get_accuracy_hb(mlp_mixer_cifar100, mixer_cifar100_hb, test_loader_mixer_cifar100, true_labels_cifar100),
    "ViT" : get_accuracy_hb(vit_cifar100, vit_cifar100_hb, test_loader_vit_cifar100, true_labels_cifar100),
    "CLIP" : get_accuracy_hb(clip_cifar100, clip_cifar100_hb, test_loader_clip_cifar100, true_labels_cifar100),
}

### Isotonic Regression

In [19]:
# Fitting Isotonic Regression
# ResNet
resnet_cifar10_ir = IsotonicRegressionCalibrator()
resnet_cifar10_ir.fit(probs_val_resnet_cifar10, labels_val_cifar10)

resnet_cifar100_ir = IsotonicRegressionCalibrator()
resnet_cifar100_ir.fit(probs_val_resnet_cifar100, labels_val_cifar100)

# MLPMixer
mixer_cifar10_ir = IsotonicRegressionCalibrator()
mixer_cifar10_ir.fit(probs_val_mixer_cifar10, labels_val_cifar10)

mixer_cifar100_ir = IsotonicRegressionCalibrator()
mixer_cifar100_ir.fit(probs_val_mixer_cifar100, labels_val_cifar100)

# ViT
vit_cifar10_ir = IsotonicRegressionCalibrator()
vit_cifar10_ir.fit(probs_val_vit_cifar10, labels_val_cifar10)

vit_cifar100_ir = IsotonicRegressionCalibrator()
vit_cifar100_ir.fit(probs_val_vit_cifar100, labels_val_cifar100)

# CLIP
clip_cifar10_ir =IsotonicRegressionCalibrator()
clip_cifar10_ir.fit(probs_val_clip_cifar10, labels_val_cifar10)

clip_cifar100_ir = IsotonicRegressionCalibrator()
clip_cifar100_ir.fit(probs_val_clip_cifar100, labels_val_cifar100)

In [20]:
def get_metrics_ir(model, calibrator, test_loader, dataset_name):
    logits = get_logits(model, test_loader, device)
    probs = torch.softmax(logits, dim=1)
    calibrated_probs = calibrator.calibrate(probs)

    if dataset_name == 'cifar10':
        metrics = calibMetrics_cifar10.get_metrics(calibrated_probs)
    else:
        metrics = calibMetrics_cifar100.get_metrics(calibrated_probs)

    return metrics

def get_accuracy_ir(model, calibrator, test_loader, true_labels):
    logits = get_logits(model, test_loader, device)
    probs = torch.softmax(logits, dim=1)
    calibrated_probs = calibrator.calibrate(probs)

    return get_accuracy_from_probs(calibrated_probs, true_labels)

In [21]:
metrics_summary_cifar10["Isotonic Regression"] = {
    "ResNet" : get_metrics_ir(resnet_cifar10, resnet_cifar10_ir, test_loader_cifar10, "cifar10"),
    "MLPMixer" : get_metrics_ir(mlp_mixer_cifar10, mixer_cifar10_ir, test_loader_mixer_cifar10, "cifar10"),
    "ViT" : get_metrics_ir(vit_cifar10, vit_cifar10_ir, test_loader_vit_cifar10, "cifar10"),
    "CLIP" : get_metrics_ir(clip_cifar10, clip_cifar10_ir, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Isotonic Regression"] = {
    "ResNet" : get_metrics_ir(resnet_cifar100, resnet_cifar100_ir, test_loader_cifar100, "cifar100"),
    "MLPMixer" : get_metrics_ir(mlp_mixer_cifar100, mixer_cifar100_ir, test_loader_mixer_cifar100, "cifar100"),
    "ViT" : get_metrics_ir(vit_cifar100, vit_cifar100_ir, test_loader_vit_cifar100, "cifar100"),
    "CLIP" :  get_metrics_ir(clip_cifar100, clip_cifar100_ir, test_loader_clip_cifar100, "cifar100"),
}

In [22]:
accuracy_summary_cifar10["Isotonic Regression"] = {
    "ResNet" : get_accuracy_ir(resnet_cifar10, resnet_cifar10_hb, test_loader_cifar10, true_labels_cifar10),
    "MLPMixer" : get_accuracy_ir(mlp_mixer_cifar10, mixer_cifar10_hb, test_loader_mixer_cifar10, true_labels_cifar10),
    "ViT" : get_accuracy_ir(vit_cifar10, vit_cifar10_hb, test_loader_vit_cifar10, true_labels_cifar10),
    "CLIP" : get_accuracy_ir(clip_cifar10, clip_cifar10_hb, test_loader_clip_cifar10, true_labels_cifar10),
}

accuracy_summary_cifar100["Isotonic Regression"] = {
    "ResNet" : get_accuracy_ir(resnet_cifar100, resnet_cifar100_hb, test_loader_cifar100, true_labels_cifar100),
    "MLPMixer" : get_accuracy_ir(mlp_mixer_cifar100, mixer_cifar100_hb, test_loader_mixer_cifar100, true_labels_cifar100),
    "ViT" : get_accuracy_ir(vit_cifar100, vit_cifar100_hb, test_loader_vit_cifar100, true_labels_cifar100),
    "CLIP" : get_accuracy_ir(clip_cifar100, clip_cifar100_hb, test_loader_clip_cifar100, true_labels_cifar100),
}

### Matrix Scaling

In [24]:
# ResNet
resnet_cifar10_ms = ModelWithMatrixScaling(resnet_cifar10, logits_dim=10, device=device)
resnet_cifar10_ms.set_matrix(val_loader_cifar10)

resnet_cifar100_ms = ModelWithMatrixScaling(resnet_cifar100, logits_dim=100, device=device)
resnet_cifar100_ms.set_matrix(val_loader_cifar100)

# MLPMixer
mixer_cifar10_ms = ModelWithMatrixScaling(mlp_mixer_cifar10, logits_dim=10, device=device)
mixer_cifar10_ms.set_matrix(val_loader_mixer_cifar10)

mixer_cifar100_ms = ModelWithMatrixScaling(mlp_mixer_cifar100, logits_dim=100, device=device)
mixer_cifar100_ms.set_matrix(val_loader_mixer_cifar100)

# ViT
vit_cifar10_ms = ModelWithMatrixScaling(vit_cifar10, logits_dim=10, device=device)
vit_cifar10_ms.set_matrix(val_loader_vit_cifar10)

vit_cifar100_ms = ModelWithMatrixScaling(vit_cifar100, logits_dim=100, device=device)
vit_cifar100_ms.set_matrix(val_loader_vit_cifar100)

# CLIP
clip_cifar10_ms = ModelWithMatrixScaling(clip_cifar10, logits_dim=10, device=device)
clip_cifar10_ms.set_matrix(val_loader_clip_cifar10)

clip_cifar100_ms = ModelWithMatrixScaling(clip_cifar100, logits_dim=100, device=device)
clip_cifar100_ms.set_matrix(val_loader_clip_cifar100)

In [25]:
metrics_summary_cifar10["Matrix Scaling"] = {
    "ResNet" : get_metrics_from_model(resnet_cifar10_ms, test_loader_cifar10, "cifar10"),
    "MLPMixer" : get_metrics_from_model(mixer_cifar10_ms, test_loader_mixer_cifar10, "cifar10"),
    "ViT" : get_metrics_from_model(vit_cifar10_ms, test_loader_vit_cifar10, "cifar10"),
    "CLIP" : get_metrics_from_model(clip_cifar10_ms, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Matrix Scaling"] = {
    "ResNet" : get_metrics_from_model(resnet_cifar100_ms, test_loader_cifar100, "cifar100"),
    "MLPMixer" : get_metrics_from_model(mixer_cifar100_ms, test_loader_mixer_cifar100, "cifar100"),
    "ViT" : get_metrics_from_model(vit_cifar100_ms, test_loader_vit_cifar100, "cifar100"),
    "CLIP" : get_metrics_from_model(clip_cifar100_ms, test_loader_clip_cifar100, "cifar100"),
}

In [26]:
accuracy_summary_cifar10["Matrix Scaling"] = {
    "ResNet" : get_accuracy_from_model(resnet_cifar10_ms, test_loader_cifar10),
    "MLPMixer" : get_accuracy_from_model(mixer_cifar10_ms, test_loader_mixer_cifar10),
    "ViT" : get_accuracy_from_model(vit_cifar10_ms, test_loader_vit_cifar10),
    "CLIP" : get_accuracy_from_model(clip_cifar10_ms, test_loader_clip_cifar10),
}

accuracy_summary_cifar100["Matrix Scaling"] = {
    "ResNet" : get_accuracy_from_model(resnet_cifar100_ms, test_loader_cifar100),
    "MLPMixer" : get_accuracy_from_model(mixer_cifar100_ms, test_loader_mixer_cifar100),
    "ViT" : get_accuracy_from_model(vit_cifar100_ms, test_loader_vit_cifar100),
    "CLIP" : get_accuracy_from_model(clip_cifar100_ms, test_loader_clip_cifar100),
}

### Vector Scaling

In [27]:
# ResNet
resnet_cifar10_vs = ModelWithVectorScaling(resnet_cifar10, logits_dim=10, device=device)
resnet_cifar10_vs.set_vector(val_loader_cifar10)

resnet_cifar100_vs = ModelWithVectorScaling(resnet_cifar100, logits_dim=100, device=device)
resnet_cifar100_vs.set_vector(val_loader_cifar100)

# MLPMixer
mixer_cifar10_vs = ModelWithVectorScaling(mlp_mixer_cifar10, logits_dim=10, device=device)
mixer_cifar10_vs.set_vector(val_loader_mixer_cifar10)

mixer_cifar100_vs = ModelWithVectorScaling(mlp_mixer_cifar100, logits_dim=100, device=device)
mixer_cifar100_vs.set_vector(val_loader_mixer_cifar100)

# ViT
vit_cifar10_vs = ModelWithVectorScaling(vit_cifar10, logits_dim=10, device=device)
vit_cifar10_vs.set_vector(val_loader_vit_cifar10)

vit_cifar100_vs = ModelWithVectorScaling(vit_cifar100, logits_dim=100, device=device)
vit_cifar100_vs.set_vector(val_loader_vit_cifar100)

# CLIP
clip_cifar10_vs = ModelWithVectorScaling(clip_cifar10, logits_dim=10, device=device)
clip_cifar10_vs.set_vector(val_loader_clip_cifar10)

clip_cifar100_vs = ModelWithVectorScaling(clip_cifar100, logits_dim=100, device=device)
clip_cifar100_vs.set_vector(val_loader_clip_cifar100)

In [28]:
metrics_summary_cifar10["Vector Scaling"] = {
    "ResNet": get_metrics_from_model(resnet_cifar10_vs, test_loader_cifar10, "cifar10"),
    "MLPMixer": get_metrics_from_model(mixer_cifar10_vs, test_loader_mixer_cifar10, "cifar10"),
    "ViT": get_metrics_from_model(vit_cifar10_vs, test_loader_vit_cifar10, "cifar10"),
    "CLIP":  get_metrics_from_model(clip_cifar10_vs, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Vector Scaling"] = {
    "ResNet": get_metrics_from_model(resnet_cifar100_vs, test_loader_cifar100, "cifar100"),
    "MLPMixer": get_metrics_from_model(mixer_cifar100_vs, test_loader_mixer_cifar100, "cifar100"),
    "ViT": get_metrics_from_model(vit_cifar100_vs, test_loader_vit_cifar100, "cifar100"),
    "CLIP": get_metrics_from_model(clip_cifar100_vs, test_loader_clip_cifar100, "cifar100"),
}

In [29]:
accuracy_summary_cifar10["Vector Scaling"] = {
    "ResNet": get_accuracy_from_model(resnet_cifar10_vs, test_loader_cifar10),
    "MLPMixer": get_accuracy_from_model(mixer_cifar10_vs, test_loader_mixer_cifar10),
    "ViT": get_accuracy_from_model(vit_cifar10_vs, test_loader_vit_cifar10),
    "CLIP":  get_accuracy_from_model(clip_cifar10_vs, test_loader_clip_cifar10),
}

accuracy_summary_cifar100["Vector Scaling"] = {
    "ResNet": get_accuracy_from_model(resnet_cifar100_vs, test_loader_cifar100),
    "MLPMixer": get_accuracy_from_model(mixer_cifar100_vs, test_loader_mixer_cifar100),
    "ViT": get_accuracy_from_model(vit_cifar100_vs, test_loader_vit_cifar100),
    "CLIP": get_accuracy_from_model(clip_cifar100_vs, test_loader_clip_cifar100),
}

### Temperature Scaling

In [30]:
# ResNet
resnet_cifar10_ts = ModelWithTemperature_CrossEntropy(resnet_cifar10, device=device)
resnet_cifar10_ts.set_temperature(val_loader_cifar10)

resnet_cifar100_ts = ModelWithTemperature_CrossEntropy(resnet_cifar100, device=device)
resnet_cifar100_ts.set_temperature(val_loader_cifar100)

# MLPMixer
mixer_cifar10_ts = ModelWithTemperature_CrossEntropy(mlp_mixer_cifar10, device=device)
mixer_cifar10_ts.set_temperature(val_loader_mixer_cifar10)

mixer_cifar100_ts = ModelWithTemperature_CrossEntropy(mlp_mixer_cifar100, device=device)
mixer_cifar100_ts.set_temperature(val_loader_mixer_cifar100)

# ViT
vit_cifar10_ts = ModelWithTemperature_CrossEntropy(vit_cifar10, device=device)
vit_cifar10_ts.set_temperature(val_loader_vit_cifar10)

vit_cifar100_ts = ModelWithTemperature_CrossEntropy(vit_cifar100, device=device)
vit_cifar100_ts.set_temperature(val_loader_vit_cifar100)

# CLIP
clip_cifar10_ts = ModelWithTemperature_CrossEntropy(clip_cifar10, device=device)
clip_cifar10_ts.set_temperature(val_loader_clip_cifar10)

clip_cifar100_ts = ModelWithTemperature_CrossEntropy(clip_cifar100, device=device)
clip_cifar100_ts.set_temperature(val_loader_clip_cifar100)

In [31]:
metrics_summary_cifar10["Temperature Scaling"] = {
    "ResNet": get_metrics_from_model(resnet_cifar10_ts, test_loader_cifar10, "cifar10"),
    "MLPMixer": get_metrics_from_model(mixer_cifar10_ts, test_loader_mixer_cifar10, "cifar10"),
    "ViT": get_metrics_from_model(vit_cifar10_ts, test_loader_vit_cifar10, "cifar10"),
    "CLIP": get_metrics_from_model(clip_cifar10_ts, test_loader_clip_cifar10, "cifar10"),
}

metrics_summary_cifar100["Temperature Scaling"] = {
    "ResNet": get_metrics_from_model(resnet_cifar100_ts, test_loader_cifar100, "cifar100"),
    "MLPMixer": get_metrics_from_model(mixer_cifar100_ts, test_loader_mixer_cifar100, "cifar100"),
    "ViT": get_metrics_from_model(vit_cifar100_ts, test_loader_vit_cifar100, "cifar100"),
    "CLIP": get_metrics_from_model(clip_cifar100_ts, test_loader_clip_cifar100, "cifar100"),
}

In [32]:
accuracy_summary_cifar10["Temperature Scaling"] = {
    "ResNet": get_accuracy_from_model(resnet_cifar10_ts, test_loader_cifar10),
    "MLPMixer": get_accuracy_from_model(mixer_cifar10_ts, test_loader_mixer_cifar10),
    "ViT": get_accuracy_from_model(vit_cifar10_ts, test_loader_vit_cifar10),
    "CLIP": get_accuracy_from_model(clip_cifar10_ts, test_loader_clip_cifar10),
}

accuracy_summary_cifar100["Temperature Scaling"] = {
    "ResNet": get_accuracy_from_model(resnet_cifar100_ts, test_loader_cifar100),
    "MLPMixer": get_accuracy_from_model(mixer_cifar100_ts, test_loader_mixer_cifar100),
    "ViT": get_accuracy_from_model(vit_cifar100_ts, test_loader_vit_cifar100),
    "CLIP": get_accuracy_from_model(clip_cifar100_ts, test_loader_clip_cifar100),
}

### Printing metrics

In [33]:
def print_metrics_summary(metrics_summary, metric = "ece"):
    table = {}

    for method, models in metrics_summary.items():
        table[method] = {}
        for model, metrics in models.items():
            table[method][model] = metrics[metric]*100

    df = pd.DataFrame(table)
    print(df)

def print_accuracy_summary(accuracy_summary):
    df = pd.DataFrame(accuracy_summary)
    df = (df * 100).round(2)
    print(df)


**Metrics Summary - ECE**

In [ ]:
print_metrics_summary(metrics_summary_cifar10, metric = "ece")

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet        3.417188           0.789930             0.860769   
MLPMixer      3.854989           1.296629             1.180978   
ViT           2.091562           0.859918             1.043529   
CLIP          5.887104           2.843945             1.316882   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet          0.805835        0.439800             0.943522  
MLPMixer        0.698081        0.739970             0.950492  
ViT             1.007215        0.792565             0.844404  
CLIP            5.648492        0.479389             1.999824  


In [37]:
print_metrics_summary(metrics_summary_cifar100, metric = "ece")

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet       14.783758           5.812093             4.890827   
MLPMixer     14.867292           7.888407             6.908596   
ViT          10.635576           6.079769             4.615878   
CLIP         10.474582           7.577950             2.997649   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet         13.769108        2.504911             4.323202  
MLPMixer       15.175846        4.969156             5.205176  
ViT            11.880732        3.318121             3.577172  
CLIP           15.088081        2.578967             2.382868  


**Metrics Summary - MCE**

In [40]:
print_metrics_summary(metrics_summary_cifar10, metric = "mce")

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet       68.099821          53.823853            70.921957   
MLPMixer     74.217099          35.495490            18.786614   
ViT          39.474577          58.752644            37.514877   
CLIP         19.464833          22.290039            18.798828   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet          9.658146       22.847486            23.707080  
MLPMixer       20.081013       76.153225            26.170030  
ViT            73.036891       29.033682            27.108955  
CLIP           17.286280       15.027705            14.357598  


In [41]:
print_metrics_summary(metrics_summary_cifar100, metric = "mce")

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet       42.566642          36.367738            11.827350   
MLPMixer     39.122123          36.556900            20.720577   
ViT          33.437470          49.600145            15.952027   
CLIP         15.847594          15.178269            10.093474   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet         37.738320        9.287119            10.658687  
MLPMixer       36.771131       15.034133            16.866124  
ViT            33.558857       12.163728            14.420056  
CLIP           22.709641        8.164650             4.435581  


**Accuracy Summary**

In [38]:
print_accuracy_summary(accuracy_summary_cifar10)

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet           95.16              95.15                95.15   
MLPMixer         93.63              93.72                93.72   
ViT              97.03              97.00                97.00   
CLIP             88.79              90.32                90.32   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet             95.58           95.50                95.16  
MLPMixer           94.17           93.83                93.63  
ViT                97.00           97.10                97.03  
CLIP               90.88           90.65                88.79  


In [39]:
print_accuracy_summary(accuracy_summary_cifar100)

          Uncalibrated  Histogram Binning  Isotonic Regression  \
ResNet           79.36              78.71                78.71   
MLPMixer         75.94              74.29                74.29   
ViT              83.65              82.71                82.71   
CLIP             61.71              62.89                62.89   

          Matrix Scaling  Vector Scaling  Temperature Scaling  
ResNet             78.21           79.63                79.36  
MLPMixer           72.36           76.22                75.94  
ViT                80.60           83.54                83.65  
CLIP               68.25           67.40                61.71  
